# Lesson: Understanding Rotary Position Embeddings (RoPE) Intuitively

Welcome! In this notebook, we are going to explore **RoPE (Rotary Position Embeddings)**—the modern positioning mechanism used in models like **Llama 3, DeepSeek, and Mirendil-Base**.

---

## 1. The Clock Analogy (Visualizing 2D Rotations)
Imagine an attention vector as a **hand on a clock face**:
- Every position step in the context window rotates the hand forward clockwise by a set angle $\theta$.
- If word A is at position $m$, its hand points at angle $m \cdot \theta$.
- If word B is at position $n$, its hand points at angle $n \cdot \theta$.
- The attention dot product $\mathbf{q}_m^\top \mathbf{k}_n$ depends **only on the angle between the two hands: $(n - m) \cdot \theta$**.

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in sys.path whether running from root or notebooks/ dir
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import math
from src.model.rope import precompute_freqs_cis, apply_rotary_emb

print(f"PyTorch version: {torch.__version__}")
print(f"Project root resolved: {root_dir}")

## 2. Interactive Experiment: Positions (10, 12) vs (100, 102)

Let's create two random vectors for two words:
- `q_word`: Query vector for `"def"`
- `k_word`: Key vector for `"quicksort"`

In both scenarios, the distance between the two words is **exactly 2 tokens**.

In [ ]:
# Setup head dimension and sequence limits
head_dim = 64
max_seq_len = 256

# Precompute rotation angles (cos and sin tables)
cos, sin = precompute_freqs_cis(dim=head_dim, max_seq_len=max_seq_len)

# Generate fixed sample vectors
torch.manual_seed(42)
q_word = torch.randn(1, 1, 1, head_dim)
k_word = torch.randn(1, 1, 1, head_dim)

### Scenario A: Early in the file (Positions 10 & 12)

In [ ]:
# Rotate q_word at position 10
q_rot_10, _ = apply_rotary_emb(q_word, q_word, cos[10:11], sin[10:11])

# Rotate k_word at position 12
_, k_rot_12 = apply_rotary_emb(k_word, k_word, cos[12:13], sin[12:13])

# Compute attention dot product
dot_product_A = (q_rot_10 * k_rot_12).sum().item()
print(f"Dot product at positions (10, 12)  [distance = 2]: {dot_product_A:.6f}")

### Scenario B: Deep in a long context (Positions 100 & 102)

In [ ]:
# Rotate q_word at position 100
q_rot_100, _ = apply_rotary_emb(q_word, q_word, cos[100:101], sin[100:101])

# Rotate k_word at position 102
_, k_rot_102 = apply_rotary_emb(k_word, k_word, cos[102:103], sin[102:103])

# Compute attention dot product
dot_product_B = (q_rot_100 * k_rot_102).sum().item()
print(f"Dot product at positions (100, 102) [distance = 2]: {dot_product_B:.6f}")

### Verification: Compare Results

In [ ]:
diff = abs(dot_product_A - dot_product_B)
print(f"Absolute difference: {diff:.10f}")
assert diff < 1e-5, "RoPE relative invariance violated!"
print("SUCCESS: RoPE relative distance invariance holds perfectly!")

## 3. What Happens When Distance Changes?
Let's test what happens when the two tokens are separated by 1, 2, 5, 10, 25, 50, 100 tokens.

In [ ]:
distances = [1, 2, 5, 10, 25, 50, 100]
print(f"{'Relative Distance':<20} | {'Dot Product Score':<20}")
print("-" * 45)

for d in distances:
    pos_q = 10
    pos_k = 10 + d
    q_rot, _ = apply_rotary_emb(q_word, q_word, cos[pos_q:pos_q+1], sin[pos_q:pos_q+1])
    _, k_rot = apply_rotary_emb(k_word, k_word, cos[pos_k:pos_k+1], sin[pos_k:pos_k+1])
    score = (q_rot * k_rot).sum().item()
    print(f"{d:<20} | {score:<20.6f}")